In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1785250991557_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [2]:
spark = SparkSession.builder \
    .appName("Yelp RAG Pipeline") \
    .getOrCreate()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [3]:
BUSINESS_PATH = "s3://yelpdatasetsmvita/gold_layer/rag_new/business_documents/"
REVIEW_PATH = "s3://yelpdatasetsmvita/gold_layer/rag_new/review_documents/"

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
business_df = spark.read.parquet(BUSINESS_PATH)
review_df = spark.read.parquet(REVIEW_PATH)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [5]:
#Verify the Schema
business_df.printSchema()

review_df.printSchema()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

root
 |-- document_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- business_name: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- primary_category: string (nullable = true)
 |-- category_list: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- business_rating: double (nullable = true)
 |-- review_count: long (nullable = true)
 |-- price_range: string (nullable = true)
 |-- is_open: integer (nullable = true)
 |-- hours: string (nullable = true)
 |-- business_features: string (nullable = true)
 |-- document_text: string (nullable = true)
 |-- document_type: string (nullable = true)
 |-- last_updated: timestamp (nullable = true)

root
 |-- document_id: string (nullable = true)
 |-- review_id: string (nullable = true)
 |-

In [6]:
#Count Records
print("Business Documents:", business_df.count())
print("Review Documents:", review_df.count())

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Business Documents: 150346
Review Documents: 6990280

In [7]:
#Display Sample Rows
business_df.select(
    "business_name",
    "city",
    "business_rating",
    "document_text"
).show(5, truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------------+------------+---------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|business_name         |city        |business_rating|document_text                                                                                                                                                         

In [8]:
review_df.select(
    "business_name",
    "stars",
    "sentiment",
    "document_text"
).show(5, truncate=False)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------------------------------------------------+-----+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [9]:
#Check for Missing Documents
#These are the most important quality checks before chunking.

from pyspark.sql.functions import col

business_df.filter(
    col("document_text").isNull()
).count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

0

In [10]:
review_df.filter(
    col("document_text").isNull()
).count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

0

In [11]:
business_df.filter(
    col("document_text") == ""
).count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

0

In [12]:
review_df.filter(
    col("document_text") == ""
).count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

0

In [13]:
#Verify Document Length
from pyspark.sql.functions import length

business_df.select(
    length("document_text").alias("length")
).describe().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+------------------+
|summary|            length|
+-------+------------------+
|  count|            150346|
|   mean| 530.7933500059862|
| stddev|101.78436320858232|
|    min|               234|
|    max|              1093|
+-------+------------------+

In [ ]:
# measuring how large each document_text is, because an embedding model cannot process unlimited text at once.

# Before we split (chunk) the documents, we need to know:

# How short are the documents?
# How long are the longest ones?
# What is the average size?

# This tells us whether chunking is necessary and helps us choose an appropriate chunk size.

In [14]:
review_df.select(
    length("document_text").alias("length")
).describe().show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+-------+-----------------+
|summary|           length|
+-------+-----------------+
|  count|          6990280|
|   mean|707.4575024176428|
| stddev|527.6846792098078|
|    min|              124|
|    max|             5194|
+-------+-----------------+

Step 2 — Document Chunking (PySpark on EMR)
Why do we need chunking?

Embedding models have token limits. Feeding an entire business profile or a very long review as one document is not ideal. Instead, we split the text into smaller chunks while preserving metadata.

we chunk both datasets

Business Documents

Yes.

Business profiles can become quite long because they include:

Business information
Categories
Features
Hours
Description

Some businesses may exceed a comfortable embedding size.

Review Documents - Mostly no.

A Yelp review is usually short (100–500 words).

Most reviews will become one chunk.

Only unusually long reviews need splitting.

We'll use the same chunking function for both datasets so the pipeline remains generic.

Chunking Strategy

We will use character-based chunking in Spark.

Why?

Fast
Distributed
No Python NLP libraries required on EMR
Easy to parallelize
Good enough for the preprocessing stage

Later, when generating embeddings, we can switch to token-aware chunking if necessary.

Chunk Size

For this project, I recommend:

Chunk Size = 1000 characters

Overlap = 200 characters

This gives:

Chunk 1
Characters 1–1000

Chunk 2
Characters 801–1800

Chunk 3
Characters 1601–2600

The overlap helps preserve context across chunk boundaries.

Output Schema

Create a new dataset called:

gold_layer/
    rag_new/
        chunked_documents/

Each row should look like:

Column	Description
chunk_id	Unique chunk identifier
document_id	Original document
business_id	Business
review_id	Only for reviews
document_type	Business / Review
chunk_number	Chunk sequence
chunk_text	Text to embed
city	Metadata
state	Metadata
primary_category	Metadata
stars	Reviews only

This metadata will later be stored alongside the embedding for filtering and retrieval.